# Servidor Único de Inferencia + Agente Ollama LLM en Google Colab (1 Clic)

Levanta en GPU T4 de Google Colab **ambos servicios en la nube**:
1. **Inferencia Unificada (`INFERENCE_URL`, puerto 8000):** M1+OCR multi-ángulo, M2 (vehículos seg), M3 (1..6+ ejes), M4 (pasajeros).
2. **Agente LLM Propio (`OLLAMA_URL`, puerto 11434):** Ollama con `llama3.2:3b` (rápido en T4, sin API keys externas).

Al ejecutar la Celda 2 te imprimirá **ambas URLs** (`INFERENCE_URL` y `OLLAMA_URL`) para pegarlas en `https://peaje-ia-web.onrender.com` -> pestaña **Estado**.

In [ ]:
!rm -rf /content/proyecto_caseta_IA
!git clone https://github.com/yeixd/proyecto_caseta_IA.git /content/proyecto_caseta_IA
%cd /content/proyecto_caseta_IA/inferencia
!pip install -q fastapi uvicorn[standard] python-multipart pydantic ultralytics easyocr opencv-python-headless python-dotenv
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!python diagnostico_entorno.py

In [ ]:
# (Opcional) Si ya tienes tus pesos entrenados .pt en Google Drive, descomenta estas 2 líneas:
# from google.colab import drive
# drive.mount('/content/drive')

import os, subprocess, time, re

OLLAMA_MODEL = 'llama3.2:3b'  # Puedes cambiar a 'llama3.1:8b' si deseas

os.environ['MODEL_M1_PLATES_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m1_placas.pt'
os.environ['MODEL_M2_VEHICLE_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m2_vehiculos_seg.pt'
os.environ['MODEL_M3_WHEEL_POSE_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m3_ruedas_pose.pt'
os.environ['MODEL_M4_PASSENGERS_SEG_PATH'] = '/content/drive/MyDrive/proyecto_peaje/models/m4_pasajeros_seg.pt'
os.environ['INFERENCE_API_KEY'] = ''
os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
os.environ['OLLAMA_ORIGINS'] = '*'

def start_tunnel(port: int):
    proc = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}', '--http-host-header', f'localhost:{port}'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    url = ''
    for line in proc.stdout:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            url = m.group(0)
            break
    return proc, url

# 1. Iniciar Ollama y descargar modelo LLM
print('Iniciando Ollama en puerto 11434...')
ollama_proc = subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(4)
print(f'Descargando modelo {OLLAMA_MODEL} (toma ~20 segundos la primera vez)...')
subprocess.run(['ollama', 'pull', OLLAMA_MODEL], check=True)

# 2. Iniciar servidor de Inferencia (M1+M2+M3+M4)
print('Iniciando servidor de Inferencia YOLO + EasyOCR en puerto 8000...')
infer_proc = subprocess.Popen(['uvicorn', 'main:app', '--host', '0.0.0.0', '--port', '8000'])
time.sleep(12)

# 3. Abrir ambos túneles HTTPS
t_infer, infer_url = start_tunnel(8000)
t_ollama, ollama_url = start_tunnel(11434)

print('\n======================================================================')
print('PEGA ESTOS 3 VALORES EN https://peaje-ia-web.onrender.com -> ESTADO:')
print('  INFERENCE_URL :', infer_url)
print('  OLLAMA_URL    :', ollama_url)
print('  Modelo Ollama :', OLLAMA_MODEL)
print('======================================================================\n')
print('Ambos servidores activos. Mantén esta celda ejecutándose.')
infer_proc.wait()